In [1]:
import sys
import os
import copy
import shutil
import cv2
import matplotlib.pyplot as plt

import numpy as np
import torch

# Add the src directory to the path. TEMPORARY FIX
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.predictor import ShorelinePredictor

from src.data_processing.dataset_loader import CoastData

from src.models.metrics import Metrics

In [2]:
# Execute this cell to make sure 
# that external modules are reloaded
%load_ext autoreload
%autoreload 2

In [3]:
image_type_paths = {
    "oblique": {
        "path": os.path.abspath(os.path.join(os.getcwd(), "../../data/processed_obliques_2_classes/")),
        "num_classes": 2,
        "weights_path": os.path.abspath(os.path.join(os.getcwd(), "../../artifacts/article/experiment2/oblique"))
    },
    # "rectified": {
    #     "path": os.path.abspath(os.path.join(os.getcwd(), "../../data/processed_rectified_3_classes/")),
    #     "num_classes": 3,
    #     "weights_path": os.path.abspath(os.path.join(os.getcwd(), "../../artifacts/article/experiment2/rectified"))
    # }
}

networks: dict[str] = {
    "UNet": {
        "weights_path": {
            "rectified": "2025-10-16-19-02-57_rectified_UNet_256x256",
            "oblique": "2025-10-14-09-58-04_oblique_UNet_256x256"
        },
        "patch": {
            "patch_size": (256, 256),
            "stride": (128, 128)
        }
    },
    "AttentionUNet": {
        "weights_path": {
            "rectified": "2025-10-16-20-56-12_rectified_AttentionUNet_256x256",
            "oblique": "2025-10-14-17-04-44_oblique_AttentionUNet_256x256"
        },
        "patch": {
            "patch_size": (256, 256),
            "stride": (128, 128)
        }
    },
    "DeepLabV3": {
        "weights_path": {
            "rectified": "2025-10-16-22-37-47_rectified_DeepLabV3_256x256",
            "oblique": "2025-10-14-23-37-23_oblique_DeepLabV3_256x256"
        },
        "patch": {
            "patch_size": (256, 256),
            "stride": (128, 128)
        }
    },
    "DuckNet": {
        "weights_path": {
            "rectified": "2025-10-17-01-46-46_rectified_DuckNet_256x256",
            "oblique": "2025-10-15-08-52-40_oblique_DuckNet_256x256"
        },
        "patch": {
            "patch_size": (256, 256),
            "stride": (128, 128)
        }
    }
}

patches = {
    "256x256": {
        "patch_size": (256, 256),
        "stride": (128, 128)
    }
}

# output_dir = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/"))

In [10]:

for data_type in image_type_paths:
    print(f"\n{'#'*30}\nProcessing {data_type} images\n{'#'*30}")
        
    data_path = image_type_paths[data_type]["path"]
    num_classes = image_type_paths[data_type]["num_classes"]
    weights_path = image_type_paths[data_type]["weights_path"]

    print(f"Data path: {data_path}")

    # Load data
    data = CoastData(data_path)

    get_mask = True
    filtered_data = data.split_data(get_metadata=True, get_mask=get_mask)

    print(f"Number of samples: {len(filtered_data['test']['images'])}")
    for network in networks:
        print(f"\n{'-'*20}\nPredicting with {network} - {data_type}\n{'-'*20}")
        net_weights_path = os.path.join(weights_path, networks[network]["weights_path"][data_type], "models/best_model.pth")

        predictor = ShorelinePredictor(network, net_weights_path, num_classes)

        counter = 0
        total_images = len(filtered_data['test']['images'])

        ignore_index = 0 if data_type == "rectified" else None
        average = 'weighted' if data_type == "rectified" else 'macro'
        metrics = Metrics(
            phase='test',
            num_classes=num_classes,
            average=average,
            compute_loss=False,
            ignore_index=ignore_index
        )

        # Predict only the test set
        for path_img, path_mask, metadata in zip(filtered_data['test']['images'], filtered_data['test']['masks'], filtered_data['test']['metadata']):
            # Get filenames
            img_filename = os.path.basename(path_img)
            mask_filename = os.path.basename(path_mask)

            gt_mask = cv2.imread(path_mask, cv2.IMREAD_GRAYSCALE)

            # Predict
            landward_pixel_pred = 1 if data_type == "rectified" else 0
            seaward_pixel_pred = 2 if data_type == "rectified" else 1
            output = predictor.predict(path_img, patch_size=networks[network]["patch"]["patch_size"], stride=networks[network]["patch"]["stride"], landward_pixel_pred=landward_pixel_pred, seaward_pixel_pred=seaward_pixel_pred)

            pred_mask = output['predicted_mask'].astype(np.uint8)
            
            # to tensor
            gt_mask_tensor = torch.tensor(gt_mask).unsqueeze(0)
            pred_mask_tensor = torch.tensor(pred_mask).unsqueeze(0)

            metrics.update_metrics(gt_mask_tensor, pred_mask_tensor)

        metrics.compute()
        print(metrics.get_last_epoch_info())



##############################
Processing oblique images
##############################
Data path: /home/josep/LOCALDATA/Shoreline-extraction/data/processed_obliques_2_classes
CoastData: global - 1717 images
Coast: agrelo, Total size: 244
Coast: arenaldentem, Total size: 40
Coast: cadiz, Total size: 946
Coast: cies, Total size: 430
Coast: samarador, Total size: 57
Number of samples: 174

--------------------
Predicting with UNet - oblique
--------------------
test metrics: 
	test_accuracy: 0.9553217887878418
	test_f1_score: 0.9553941488265991
	test_precision: 0.955748438835144
	test_recall: 0.9553217887878418
	test_confusion_matrix: 
		0.9429 0.0571
		0.0322 0.9678


--------------------
Predicting with AttentionUNet - oblique
--------------------
test metrics: 
	test_accuracy: 0.948209822177887
	test_f1_score: 0.9481617212295532
	test_precision: 0.9481180906295776
	test_recall: 0.948209822177887
	test_confusion_matrix: 
		0.9480 0.0520
		0.0516 0.9484


--------------------
Predictin